# 🏆 HalfKP NNUE Training (Resume & RAM Optimized)

## ✨ Features
- ✅ **Auto-Resume**: Lanjutkan training dari checkpoint terakhir
- ✅ **Hemat RAM**: Streaming data dengan memmap (<10GB usage)
- ✅ **Checkpoint Setiap Epoch**: Tidak kehilangan progress
- ✅ **Kaggle Output Persistence**: Simpan ke /kaggle/working
- ✅ **Deterministic Split**: Train/Val konsisten antar session


In [1]:
!pip install python-chess tqdm matplotlib --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 65.8 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR
import numpy as np
import chess
import os
import time
import struct
import math
import json
import gc
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_cuda = device.type == 'cuda'

if use_cuda:
    props = torch.cuda.get_device_properties(0)
    print(f"🖥️ GPU: {props.name}, VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU, using CPU")

print(f"🔧 PyTorch: {torch.__version__}")
print(f"💾 RAM Check: {torch.cuda.memory_allocated()/1e9:.2f} GB used" if use_cuda else "CPU mode")

🖥️ GPU: Tesla T4, VRAM: 15.8 GB
🔧 PyTorch: 2.8.0+cu126
💾 RAM Check: 0.00 GB used


## 📋 Configuration

In [3]:
class Config:
    # === DATA ===
    DATA_PATH = "/kaggle/input/data-train-nnue/training.binpack"
    MAX_SAMPLES = 0  # 0 = all

    # === NETWORK ===
    L1_SIZE = 256
    L2_SIZE = 32
    OUTPUT_SIZE = 1

    # === TRAINING ===
    EPOCHS = 50
    BATCH_SIZE = 16384 if use_cuda else 4096
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 0.01
    GRAD_CLIP = 1.0

    # === LAMBDA SCHEDULE ===
    LAMBDA_START = 0.5
    LAMBDA_END = 0.9

    # === LR SCHEDULE ===
    LR_WARMUP_EPOCHS = 5
    LR_MIN_RATIO = 0.01

    # === EARLY STOPPING ===
    EARLY_STOPPING = True
    PATIENCE = 15
    MIN_DELTA = 0.0001

    # === CHECKPOINT (PENTING!) ===
    CHECKPOINT_LOAD_DIR = "/kaggle/input/data-train-nnue"  # <-- Ubah sesuai lokasi checkpoint lama
    # Untuk SAVE checkpoint baru (harus writable, gunakan /kaggle/working)
    CHECKPOINT_SAVE_DIR = "/kaggle/working/checkpoints"
    CHECKPOINT_EVERY = 1  # Save setiap epoch!
    AUTO_RESUME = True    # Auto-load checkpoint terakhir

    # === QUANTIZATION ===
    INPUT_QUANT_SCALE = 127
    HIDDEN_QUANT_SCALE = 64
    OUTPUT_SCALE = 400
    SCORE_CLAMP = 10000

    # === OUTPUT ===
    OUTPUT_PATH = "/kaggle/working/model.nnue"

    # === GPU OPTIMIZATION ===
    USE_AMP = use_cuda
    PIN_MEMORY = use_cuda
    NUM_WORKERS = (os.cpu_count() or 1) if use_cuda else 0  # Use ALL CPU cores

    # === MEMORY OPTIMIZATION ===
    PREFETCH_FACTOR = 2
    DETERMINISTIC_SEED = 42  # Untuk split konsisten

cfg = Config()

# Create checkpoint directory
os.makedirs(cfg.CHECKPOINT_SAVE_DIR, exist_ok=True)

# Constants
NUM_SQUARES = 64
NUM_PIECE_TYPES = 10
HALFKP_FEATURES = 64 * 10 * 64  # 40960

print(f"✅ Config loaded")
print(f"📂 Checkpoint LOAD from: {cfg.CHECKPOINT_LOAD_DIR}")
print(f"💾 Checkpoint SAVE to: {cfg.CHECKPOINT_SAVE_DIR}")
print(f"🔄 Auto-Resume: {cfg.AUTO_RESUME}")
print(f"🖥️ CPU cores available: {os.cpu_count()}")
print(f"👷 DataLoader workers: {cfg.NUM_WORKERS}")

✅ Config loaded
📂 Checkpoint LOAD from: /kaggle/input/data-train-nnue
💾 Checkpoint SAVE to: /kaggle/working/checkpoints
🔄 Auto-Resume: True
🖥️ CPU cores available: 4
👷 DataLoader workers: 4


## 🧠 Network

In [4]:
class ClippedReLU(nn.Module):
    def forward(self, x):
        return torch.clamp(x, 0.0, 1.0)

class SimpleHalfKPNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.ft_weight = nn.Embedding(HALFKP_FEATURES, cfg.L1_SIZE)
        self.ft_bias = nn.Parameter(torch.zeros(cfg.L1_SIZE))
        self.l1 = nn.Linear(cfg.L1_SIZE * 2, cfg.L2_SIZE)
        self.l2 = nn.Linear(cfg.L2_SIZE, cfg.OUTPUT_SIZE)
        self.clipped_relu = ClippedReLU()

        nn.init.normal_(self.ft_weight.weight, 0, 0.01)
        nn.init.kaiming_normal_(self.l1.weight, mode='fan_out', nonlinearity='relu')
        nn.init.zeros_(self.l1.bias)
        nn.init.xavier_normal_(self.l2.weight)
        nn.init.zeros_(self.l2.bias)

    def forward(self, white_features, black_features, white_counts, black_counts, stm):
        batch_size = white_features.size(0)
        max_features = white_features.size(1)

        feature_indices = torch.arange(max_features, device=white_features.device)
        w_valid = (feature_indices.unsqueeze(0) < white_counts.unsqueeze(1)).float().unsqueeze(2)
        b_valid = (feature_indices.unsqueeze(0) < black_counts.unsqueeze(1)).float().unsqueeze(2)

        w_indices = white_features.clamp(0, HALFKP_FEATURES - 1)
        b_indices = black_features.clamp(0, HALFKP_FEATURES - 1)

        w_embeds = self.ft_weight(w_indices)
        b_embeds = self.ft_weight(b_indices)

        acc_white = (w_embeds * w_valid).sum(dim=1) + self.ft_bias
        acc_black = (b_embeds * b_valid).sum(dim=1) + self.ft_bias

        acc_white = self.clipped_relu(acc_white)
        acc_black = self.clipped_relu(acc_black)

        stm_mask = (stm == 0).unsqueeze(1).float()
        combined = torch.cat([
            acc_white * stm_mask + acc_black * (1 - stm_mask),
            acc_black * stm_mask + acc_white * (1 - stm_mask)
        ], dim=1)

        x = self.l1(combined)
        x = self.clipped_relu(x)
        x = self.l2(x)
        return x.squeeze(-1)

print(f"✅ Network defined")

✅ Network defined


## 📦 Memory-Efficient Dataset (Streaming)

In [5]:
def piece_to_index(piece):
    if piece is None or piece.piece_type == chess.KING:
        return -1
    base = piece.piece_type - 1
    if piece.color == chess.BLACK:
        base += 5
    return base

def get_halfkp_features(board):
    white_king_sq = board.king(chess.WHITE)
    black_king_sq = board.king(chess.BLACK)
    if white_king_sq is None or black_king_sq is None:
        return None

    black_king_sq_mirrored = chess.square_mirror(black_king_sq)
    white_features = []
    black_features = []

    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if piece is None or piece.piece_type == chess.KING:
            continue
        piece_idx = piece_to_index(piece)
        if piece_idx < 0:
            continue
        white_feat = white_king_sq * 640 + piece_idx * 64 + sq
        white_features.append(white_feat)
        sq_mirrored = chess.square_mirror(sq)
        piece_idx_black = (piece_idx + 5) % 10
        black_feat = black_king_sq_mirrored * 640 + piece_idx_black * 64 + sq_mirrored
        black_features.append(black_feat)

    return white_features, black_features


class StreamingBinpackDataset(Dataset):
    """Memory-efficient streaming dataset using memmap."""

    ENTRY_SIZE = 40
    PIECE_MAP = {1:'P', 2:'N', 3:'B', 4:'R', 5:'Q', 6:'K',
                 7:'p', 8:'n', 9:'b', 10:'r', 11:'q', 12:'k'}

    def __init__(self, filepath, indices=None, max_samples=0):
        self.filepath = filepath
        self.max_features = 32

        file_size = os.path.getsize(filepath)
        total = file_size // self.ENTRY_SIZE
        
#       Before 
#       self.data = np.memmap(filepath, dtype=np.uint8, mode='r', shape=(total, self.ENTRY_SIZE))
        # Use memmap for memory efficiency
        print(f"📥 Loading entire dataset to RAM ({file_size/1e9:.2f} GB)...")
        with open(filepath, 'rb') as f:
            self.data = np.frombuffer(f.read(), dtype=np.uint8).reshape(total, self.ENTRY_SIZE)

        if indices is not None:
            self.indices = indices
        else:
            num = min(max_samples, total) if max_samples > 0 else total
            self.indices = np.arange(num)

        print(f"📊 Dataset: {len(self.indices):,} samples (streaming)")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        entry = self.data[real_idx]

        board = chess.Board.empty()
        for sq in range(64):
            byte_idx = sq // 2
            piece_code = (entry[byte_idx] & 0x0F) if sq % 2 == 0 else ((entry[byte_idx] >> 4) & 0x0F)
            if 0 < piece_code <= 12:
                board.set_piece_at(sq, chess.Piece.from_symbol(self.PIECE_MAP[piece_code]))

        stm = entry[32]
        result_code = entry[36]
        score = float(np.frombuffer(entry[38:40].tobytes(), dtype=np.int16)[0])
        score = max(-cfg.SCORE_CLAMP, min(cfg.SCORE_CLAMP, score))

        board.turn = chess.WHITE if stm == 0 else chess.BLACK
        result = 1.0 if result_code == 2 else (0.0 if result_code == 0 else 0.5)

        features = get_halfkp_features(board)

        if features is None:
            return self._empty_sample()

        white_feats, black_feats = features
        w_count = min(len(white_feats), self.max_features)
        b_count = min(len(black_feats), self.max_features)
        white_feats = (white_feats + [0] * self.max_features)[:self.max_features]
        black_feats = (black_feats + [0] * self.max_features)[:self.max_features]

        return {
            'white_features': torch.tensor(white_feats, dtype=torch.long),
            'black_features': torch.tensor(black_feats, dtype=torch.long),
            'white_count': torch.tensor(w_count, dtype=torch.long),
            'black_count': torch.tensor(b_count, dtype=torch.long),
            'stm': torch.tensor(stm, dtype=torch.long),
            'score': torch.tensor(score, dtype=torch.float32),
            'result': torch.tensor(result, dtype=torch.float32),
        }

    def _empty_sample(self):
        return {
            'white_features': torch.zeros(self.max_features, dtype=torch.long),
            'black_features': torch.zeros(self.max_features, dtype=torch.long),
            'white_count': torch.tensor(0, dtype=torch.long),
            'black_count': torch.tensor(0, dtype=torch.long),
            'stm': torch.tensor(0, dtype=torch.long),
            'score': torch.tensor(0.0, dtype=torch.float32),
            'result': torch.tensor(0.5, dtype=torch.float32),
        }

print("✅ Dataset class defined")

✅ Dataset class defined


## 💾 Checkpoint System

In [6]:
def get_save_checkpoint_path(epoch=None):
    """Get path for SAVING checkpoints (writable directory)."""
    if epoch is not None:
        return os.path.join(cfg.CHECKPOINT_SAVE_DIR, f"checkpoint_epoch_{epoch:03d}.pt")
    return os.path.join(cfg.CHECKPOINT_SAVE_DIR, "checkpoint_latest.pt")


def save_checkpoint(epoch, model, optimizer, scheduler, history, best_loss, best_error, early_stopper):
    """Save complete training state."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history,
        'best_loss': best_loss,
        'best_error': best_error,
        'early_stopper': {
            'counter': early_stopper.counter,
            'best_loss': early_stopper.best_loss,
            'should_stop': early_stopper.should_stop,
        },
        'config': {
            'L1_SIZE': cfg.L1_SIZE,
            'L2_SIZE': cfg.L2_SIZE,
            'LAMBDA_START': cfg.LAMBDA_START,
            'LAMBDA_END': cfg.LAMBDA_END,
            'EPOCHS': cfg.EPOCHS,
        }
    }

   # Save checkpoint to SAVE directory
    path = get_save_checkpoint_path()
    torch.save(checkpoint, path)

    # Also save epoch-specific
    epoch_path = get_save_checkpoint_path(epoch)
    torch.save(checkpoint, epoch_path)

    print(f"   💾 Checkpoint saved: epoch {epoch+1} -> {cfg.CHECKPOINT_SAVE_DIR}")

def find_latest_checkpoint():
    """
    Find the latest checkpoint to resume from.
    Priority:
    1. First check SAVE directory (from current/recent sessions)
    2. Then check LOAD directory (from previous sessions, e.g., Kaggle input)
    """
    # Search in both directories, prioritize SAVE dir (more recent)
    search_dirs = [
        cfg.CHECKPOINT_SAVE_DIR,  # Priority 1: current session checkpoints
        cfg.CHECKPOINT_LOAD_DIR,  # Priority 2: previous session checkpoints
    ]

    for checkpoint_dir in search_dirs:
        if not os.path.exists(checkpoint_dir):
            continue

        # Check for latest checkpoint
        latest_path = os.path.join(checkpoint_dir, "checkpoint_latest.pt")
        if os.path.exists(latest_path):
            return latest_path

        # Fallback: find highest epoch checkpoint in this directory
        checkpoints = []
        for f in os.listdir(checkpoint_dir):
            if f.startswith('checkpoint_epoch_') and f.endswith('.pt'):
                epoch = int(f.split('_')[-1].replace('.pt', ''))
                checkpoints.append((epoch, os.path.join(checkpoint_dir, f)))

        if checkpoints:
            checkpoints.sort(reverse=True)
            return checkpoints[0][1]

    return None

def load_checkpoint(model, optimizer, scheduler):
    """Load checkpoint if available."""
    checkpoint_path = find_latest_checkpoint()

    if checkpoint_path is None:
        print("📭 No checkpoint found, starting fresh")
        return 0, {'train_loss': [], 'val_loss': [], 'train_wdl': [], 'val_wdl': [],
                   'train_score': [], 'val_score': [], 'score_error': [],
                   'wdl_accuracy': [], 'lr': [], 'lambda': []}, float('inf'), float('inf'), None

    print(f"📂 Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint['history']
    best_loss = checkpoint['best_loss']
    best_error = checkpoint['best_error']
    early_stopper_state = checkpoint.get('early_stopper')

    print(f"✅ Resumed from epoch {start_epoch}")
    print(f"   Best loss: {best_loss:.4f}, Best error: {best_error:.1f}cp")

    return start_epoch, history, best_loss, best_error, early_stopper_state

print("✅ Checkpoint system ready")

✅ Checkpoint system ready


## 🎯 Loss & Training Functions

In [7]:
def get_lambda(epoch, total_epochs):
    progress = epoch / max(total_epochs - 1, 1)
    return cfg.LAMBDA_START + progress * (cfg.LAMBDA_END - cfg.LAMBDA_START)

def get_lr_multiplier(epoch, total_epochs):
    warmup = cfg.LR_WARMUP_EPOCHS
    if epoch < warmup:
        return (epoch + 1) / warmup
    progress = (epoch - warmup) / max(total_epochs - warmup - 1, 1)
    return cfg.LR_MIN_RATIO + (1 - cfg.LR_MIN_RATIO) * (1 + math.cos(math.pi * progress)) / 2

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.should_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop

    def load_state(self, state):
        if state:
            self.counter = state['counter']
            self.best_loss = state['best_loss']
            self.should_stop = state['should_stop']

# AMP setup
scaler = torch.amp.GradScaler('cuda') if cfg.USE_AMP else None

def amp_autocast():
    if cfg.USE_AMP:
        return torch.amp.autocast('cuda')
    import contextlib
    return contextlib.nullcontext()

def wdl_loss(pred, target_score, target_result, lambda_=0.5):
    pred_logits = pred / cfg.OUTPUT_SCALE
    score_logits = target_score / cfg.OUTPUT_SCALE
    wdl = F.binary_cross_entropy_with_logits(pred_logits, target_result, reduction='mean')
    pred_prob = torch.sigmoid(pred_logits)
    score_prob = torch.sigmoid(score_logits)
    score_loss = F.mse_loss(pred_prob, score_prob, reduction='mean')
    total = lambda_ * score_loss + (1 - lambda_) * wdl
    return total, wdl.item(), score_loss.item()

def train_epoch(model, dataloader, optimizer, lambda_=0.5):
    model.train()
    total_loss, total_wdl, total_score, num_batches = 0, 0, 0, 0

    for batch in tqdm(dataloader, desc="Training", leave=False):
        white_features = batch['white_features'].to(device, non_blocking=True)
        black_features = batch['black_features'].to(device, non_blocking=True)
        white_counts = batch['white_count'].to(device, non_blocking=True)
        black_counts = batch['black_count'].to(device, non_blocking=True)
        stm = batch['stm'].to(device, non_blocking=True)
        score = batch['score'].to(device, non_blocking=True)
        result = batch['result'].to(device, non_blocking=True)

        if white_counts.sum() == 0:
            continue

        optimizer.zero_grad(set_to_none=True)

        with amp_autocast():
            pred = model(white_features, black_features, white_counts, black_counts, stm)
            loss, wdl_val, score_val = wdl_loss(pred, score, result, lambda_)

        if torch.isnan(loss):
            continue

        if cfg.USE_AMP and scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

        total_loss += loss.item()
        total_wdl += wdl_val
        total_score += score_val
        num_batches += 1

    return {'loss': total_loss / max(num_batches, 1),
            'wdl_loss': total_wdl / max(num_batches, 1),
            'score_loss': total_score / max(num_batches, 1)}

@torch.no_grad()
def validate(model, dataloader, lambda_=0.5):
    model.eval()
    total_loss, total_wdl, total_score, total_error = 0, 0, 0, 0
    total_wdl_correct, total_samples, num_batches = 0, 0, 0

    for batch in tqdm(dataloader, desc="Validating", leave=False):
        white_features = batch['white_features'].to(device, non_blocking=True)
        black_features = batch['black_features'].to(device, non_blocking=True)
        white_counts = batch['white_count'].to(device, non_blocking=True)
        black_counts = batch['black_count'].to(device, non_blocking=True)
        stm = batch['stm'].to(device, non_blocking=True)
        score = batch['score'].to(device, non_blocking=True)
        result = batch['result'].to(device, non_blocking=True)

        if white_counts.sum() == 0:
            continue

        with amp_autocast():
            pred = model(white_features, black_features, white_counts, black_counts, stm)
            loss, wdl_val, score_val = wdl_loss(pred, score, result, lambda_)

        if torch.isnan(loss):
            continue

        error = torch.abs(pred - score).mean().item()
        pred_wdl = torch.sigmoid(pred / cfg.OUTPUT_SCALE)
        pred_class = (pred_wdl > 0.55).float() - (pred_wdl < 0.45).float()
        target_class = (result > 0.55).float() - (result < 0.45).float()
        wdl_correct = (pred_class == target_class).float().sum().item()

        total_loss += loss.item()
        total_wdl += wdl_val
        total_score += score_val
        total_error += error
        total_wdl_correct += wdl_correct
        total_samples += result.size(0)
        num_batches += 1

    return {'loss': total_loss / max(num_batches, 1),
            'wdl_loss': total_wdl / max(num_batches, 1),
            'score_loss': total_score / max(num_batches, 1),
            'score_error': total_error / max(num_batches, 1),
            'wdl_accuracy': total_wdl_correct / max(total_samples, 1) * 100}

print("✅ Training functions ready")

✅ Training functions ready


## 📤 Export Function

In [8]:
def export_network(model, filepath):
    model_to_export = model._orig_mod if hasattr(model, '_orig_mod') else model

    with open(filepath, 'wb') as f:
        f.write(b'NNUE')
        f.write(struct.pack('<I', 1))
        f.write(struct.pack('<I', 0))

        ft_weight = model_to_export.ft_weight.weight.detach().cpu().numpy()
        ft_weight_q = np.clip(ft_weight * cfg.INPUT_QUANT_SCALE, -32768, 32767).astype(np.int16)
        f.write(ft_weight_q.tobytes())

        ft_bias = model_to_export.ft_bias.detach().cpu().numpy()
        ft_bias_q = np.clip(ft_bias * cfg.INPUT_QUANT_SCALE, -32768, 32767).astype(np.int16)
        f.write(ft_bias_q.tobytes())

        l1_weight = model_to_export.l1.weight.detach().cpu().numpy()
        l1_weight_q = np.clip(l1_weight * cfg.HIDDEN_QUANT_SCALE, -128, 127).astype(np.int8)
        f.write(l1_weight_q.tobytes())

        l1_bias = model_to_export.l1.bias.detach().cpu().numpy()
        l1_bias_q = np.clip(l1_bias * cfg.INPUT_QUANT_SCALE * cfg.HIDDEN_QUANT_SCALE,
                            -2147483648, 2147483647).astype(np.int32)
        f.write(l1_bias_q.tobytes())

        l2_weight = model_to_export.l2.weight.detach().cpu().numpy()
        l2_weight_q = np.clip(l2_weight * cfg.HIDDEN_QUANT_SCALE, -128, 127).astype(np.int8)
        f.write(l2_weight_q.tobytes())

        l2_bias = model_to_export.l2.bias.detach().cpu().numpy()
        l2_bias_q = np.clip(l2_bias * cfg.INPUT_QUANT_SCALE * cfg.HIDDEN_QUANT_SCALE * cfg.HIDDEN_QUANT_SCALE,
                            -2147483648, 2147483647).astype(np.int32)
        f.write(l2_bias_q.tobytes())

    print(f"✓ Exported to {filepath}")

print("✅ Export function ready")

✅ Export function ready


## 📂 Load Data (Deterministic Split)

In [9]:
# Load with deterministic split for consistency across sessions
print(f"📂 Loading: {cfg.DATA_PATH}")

file_size = os.path.getsize(cfg.DATA_PATH)
total_entries = file_size // 40
num_samples = min(cfg.MAX_SAMPLES, total_entries) if cfg.MAX_SAMPLES > 0 else total_entries

print(f"📊 Total entries: {total_entries:,}")
print(f"📊 Using samples: {num_samples:,}")

# Deterministic split (same across sessions!)
np.random.seed(cfg.DETERMINISTIC_SEED)
all_indices = np.random.permutation(num_samples)
train_size = int(0.9 * num_samples)
train_indices = all_indices[:train_size]
val_indices = all_indices[train_size:]

print(f"🎯 Train: {len(train_indices):,}, Val: {len(val_indices):,}")

# Create datasets
train_dataset = StreamingBinpackDataset(cfg.DATA_PATH, indices=train_indices)
val_dataset = StreamingBinpackDataset(cfg.DATA_PATH, indices=val_indices)

# Create dataloaders
train_loader = DataLoader(
    train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
    num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY,
    persistent_workers=cfg.NUM_WORKERS > 0,
    prefetch_factor=cfg.PREFETCH_FACTOR if cfg.NUM_WORKERS > 0 else None
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False,
    num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY,
    persistent_workers=cfg.NUM_WORKERS > 0,
    prefetch_factor=cfg.PREFETCH_FACTOR if cfg.NUM_WORKERS > 0 else None
)

print(f"📦 Batches - Train: {len(train_loader):,}, Val: {len(val_loader):,}")

# Force garbage collection
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

📂 Loading: /kaggle/input/data-train-nnue/training.binpack
📊 Total entries: 107,109,331
📊 Using samples: 107,109,331
🎯 Train: 96,398,397, Val: 10,710,934
📥 Loading entire dataset to RAM (4.28 GB)...
📊 Dataset: 96,398,397 samples (streaming)
📥 Loading entire dataset to RAM (4.28 GB)...
📊 Dataset: 10,710,934 samples (streaming)
📦 Batches - Train: 5,884, Val: 654


## 🚀 Training Loop (with Resume)

In [10]:
# Initialize model
model = SimpleHalfKPNetwork().to(device)
optimizer = optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
lr_lambda = lambda epoch: get_lr_multiplier(epoch, cfg.EPOCHS)
scheduler = LambdaLR(optimizer, lr_lambda)
early_stopper = EarlyStopping(patience=cfg.PATIENCE, min_delta=cfg.MIN_DELTA)

# Try to resume from checkpoint
start_epoch = 0
best_loss = float('inf')
best_score_error = float('inf')

if cfg.AUTO_RESUME:
    start_epoch, history, best_loss, best_score_error, es_state = load_checkpoint(
        model, optimizer, scheduler
    )
    early_stopper.load_state(es_state)
else:
    history = {'train_loss': [], 'val_loss': [], 'train_wdl': [], 'val_wdl': [],
               'train_score': [], 'val_score': [], 'score_error': [],
               'wdl_accuracy': [], 'lr': [], 'lambda': []}

print(f"\n🚀 Training: epochs {start_epoch+1} to {cfg.EPOCHS}")
print(f"   Lambda: {cfg.LAMBDA_START} → {cfg.LAMBDA_END}")
print(f"   Checkpoint every: {cfg.CHECKPOINT_EVERY} epoch(s)")

📂 Loading checkpoint: /kaggle/input/data-train-nnue/checkpoint_epoch_014.pt
✅ Resumed from epoch 15
   Best loss: 0.2557, Best error: 172.2cp

🚀 Training: epochs 16 to 50
   Lambda: 0.5 → 0.9
   Checkpoint every: 1 epoch(s)


In [ ]:
# Main training loop
for epoch in range(start_epoch, cfg.EPOCHS):
    start_time = time.time()

    current_lambda = get_lambda(epoch, cfg.EPOCHS)
    current_lr = optimizer.param_groups[0]['lr']

    train_stats = train_epoch(model, train_loader, optimizer, current_lambda)
    val_stats = validate(model, val_loader, current_lambda)

    scheduler.step()

    # Record history
    history['train_loss'].append(train_stats['loss'])
    history['val_loss'].append(val_stats['loss'])
    history['train_wdl'].append(train_stats['wdl_loss'])
    history['val_wdl'].append(val_stats['wdl_loss'])
    history['train_score'].append(train_stats['score_loss'])
    history['val_score'].append(val_stats['score_loss'])
    history['score_error'].append(val_stats['score_error'])
    history['wdl_accuracy'].append(val_stats['wdl_accuracy'])
    history['lr'].append(current_lr)
    history['lambda'].append(current_lambda)

    elapsed = time.time() - start_time

    # Check for best model
    is_best = False
    if val_stats['score_error'] < best_score_error:
        best_score_error = val_stats['score_error']
        is_best = True
    if val_stats['loss'] < best_loss:
        best_loss = val_stats['loss']
        is_best = True

    status = "⭐ BEST" if is_best else ""
    print(f"Epoch {epoch+1:3d}/{cfg.EPOCHS} | "
          f"Train: {train_stats['loss']:.4f} | Val: {val_stats['loss']:.4f} | "
          f"Error: {val_stats['score_error']:.1f}cp | WDL: {val_stats['wdl_accuracy']:.1f}% | "
          f"λ: {current_lambda:.2f} | {elapsed:.1f}s {status}")

    # Save best model
    if is_best:
        export_network(model, cfg.OUTPUT_PATH)
        torch.save(model.state_dict(), cfg.OUTPUT_PATH.replace('.nnue', '.pt'))

    # Save checkpoint
    if (epoch + 1) % cfg.CHECKPOINT_EVERY == 0:
        save_checkpoint(epoch, model, optimizer, scheduler, history,
                       best_loss, best_score_error, early_stopper)

    # Early stopping
    if cfg.EARLY_STOPPING and early_stopper(val_stats['loss']):
        print(f"\n⚠️ Early stopping! No improvement for {cfg.PATIENCE} epochs.")
        save_checkpoint(epoch, model, optimizer, scheduler, history,
                       best_loss, best_score_error, early_stopper)
        break

    # Garbage collection
    if (epoch + 1) % 5 == 0:
        gc.collect()
        if use_cuda:
            torch.cuda.empty_cache()

print(f"\n✅ Training complete!")
print(f"   Best val loss: {best_loss:.4f}")
print(f"   Best score error: {best_score_error:.1f}cp")

Training:   0%|          | 0/5884 [00:12<?, ?it/s]

## 📊 Final Summary

In [ ]:
import matplotlib.pyplot as plt

if len(history['train_loss']) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0, 0].plot(history['train_loss'], label='Train')
    axes[0, 0].plot(history['val_loss'], label='Val')
    axes[0, 0].set_title('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(history['score_error'], color='orange')
    axes[0, 1].set_title('Score Error (cp)')
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(history['wdl_accuracy'], color='green')
    axes[1, 0].set_title('WDL Accuracy (%)')
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(history['lr'], color='purple')
    axes[1, 1].set_title('Learning Rate')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/kaggle/working/training_history.png', dpi=150)
    plt.show()

# List outputs
print("\n📁 Output files:")
for f in os.listdir('/kaggle/working'):
    if not f.startswith('.'):
        path = f'/kaggle/working/{f}'
        if os.path.isfile(path):
            size = os.path.getsize(path) / 1024
            print(f"  {f}: {size:.1f} KB")